# SpaceX Falcon 9 landing analysis

A reproducible companion to the final project report. The analysis uses the IBM Skills Network course snapshots in this repository; it does not present them as a fresh live API pull.

In [ ]:
from pathlib import Path
import json
import sqlite3
import pandas as pd
import numpy as np
ROOT = Path.cwd()
launches = pd.read_csv(ROOT / 'dataset_part_2.csv')
features = pd.read_csv(ROOT / 'dataset_part_3.csv')
dashboard = pd.read_csv(ROOT / 'spacex_launch_dash.csv')
sql_data = pd.read_csv(ROOT / 'Spacex.csv')
print({name: frame.shape for name, frame in [('launches', launches), ('features', features), ('dashboard', dashboard), ('sql', sql_data)]})

## Clean and inspect the released launch data

Check duplicate flight numbers, missing values, and the binary landing outcome before interpreting rates.

In [ ]:
print('Duplicate flight numbers:', launches['FlightNumber'].duplicated().sum())
print('Landing outcome counts:')
print(launches['Class'].value_counts(dropna=False))
print('Most missing fields:')
print(launches.isna().sum().sort_values(ascending=False).head(10))

## Exploratory data analysis

The grouped statistics below produce the launch-site and year comparisons in the presentation.

In [ ]:
launches['Year'] = pd.to_datetime(launches['Date']).dt.year
site_stats = launches.groupby('LaunchSite')['Class'].agg(['count', 'sum', 'mean'])
year_stats = launches.groupby('Year')['Class'].agg(['count', 'sum', 'mean'])
display(site_stats, year_stats)

In [ ]:
ax = year_stats['mean'].plot(kind='bar', figsize=(10, 4), color='#287e9b', ylim=(0, 1), title='Landing success rate by year')
ax.set_ylabel('Success rate')

## SQL checks

The course SQL snapshot is loaded into in-memory SQLite. These are examples from the ten SQL questions summarized in the report.

In [ ]:
connection = sqlite3.connect(':memory:')
sql_data.to_sql('SPACEXTBL', connection, index=False)
print(pd.read_sql_query('SELECT COUNT(*) AS Launches FROM SPACEXTBL', connection))
print(pd.read_sql_query("SELECT Launch_Site, COUNT(*) AS Launches FROM SPACEXTBL GROUP BY Launch_Site ORDER BY Launches DESC", connection))

## Predictive modeling

The full source in `analyze.py` compares logistic regression, a linear margin classifier, k-nearest neighbors, and a shallow decision tree. It uses a stratified 80/20 train/test split and five-fold cross-validation for model selection.

In [ ]:
import runpy
runpy.run_path(str(ROOT / 'analyze.py'), run_name='__main__')
results = json.loads((ROOT / 'analysis_results.json').read_text(encoding='utf-8'))
print('Dataset summary:', results['dataset'])
print('Model results:', results.get('models', results.get('classification', 'See analysis_results.json')))

## Interactive outputs

Run `visuals.py` to regenerate `launch_sites_map.html` and `launch_dashboard.html`. Open those files in a browser to inspect site-level Folium markers and the Plotly dashboard. The final PDF summarizes their results and the model limitations.